# 🗺️ Enriquecimento Geográfico do CPGF por UF da Unidade Gestora

## POC independente — Versão 1.1

Esta versão preserva integralmente a dimensão UG→UF validada na V1.0 e corrige a principal limitação encontrada na execução real: **a data da transação não está publicamente observável em parcela relevante das operações sob sigilo**.

Por isso, a V1.1 adota **duas referências temporais separadas**, sem criar ano híbrido:

### Visão A — Ano da transação

```text
REFERENCIA_TEMPORAL = TRANSACAO
ANO = ANO_TRANSACAO
```

Finalidade:

- comportamento temporal;
- compras e saques observáveis;
- integração com as trilhas analíticas;
- métricas em que a data efetiva da operação é necessária.

Limitação:

> só entram registros com `DATA TRANSAÇÃO` observável.

### Visão B — Ano do extrato

```text
REFERENCIA_TEMPORAL = EXTRATO
ANO = ANO_EXTRATO_REF
```

Finalidade:

- cobertura temporal integral por exercício do extrato;
- valor total registrado;
- valor e percentual sob sigilo;
- observabilidade dos dados públicos.

A V1.1 **não mistura** `ANO_TRANSACAO` e `ANO_EXTRATO` em um único indicador.

---

## 🔒 O que permanece inalterado

- Regras T01–T09: `1.2.0`;
- Motor/Governança: `1.3.2`;
- CSV bruto do CPGF;
- chave territorial `UG_ID` com seis dígitos;
- cinco complementos manuais com proveniência explícita;
- interpretação da UF como **localização cadastral da Unidade Gestora**, e não local físico da transação.

---

## 🆕 Principais mudanças da V1.1

1. `VALOR_TOTAL_TRANSACIONADO` passa a se chamar `VALOR_TRANSACIONADO_OBSERVAVEL` na visão por transação.
2. `N_UG_ATIVAS` passa a `N_UG_COM_MOVIMENTACAO`.
3. criação de `agg_cpgf_uf_ano_transacao`.
4. criação de `agg_cpgf_uf_ano_extrato`.
5. criação de métricas de **sigilo e observabilidade** na visão por extrato.
6. criação de tabela semântica longa para o dashboard.
7. Folium com seletor de **referência temporal + ano + métrica**.
8. metadados da dimensão incluem versão/tipo da fonte cadastral.

In [ ]:
# ============================================================
# 📦 DEPENDÊNCIAS
# ============================================================

import sys
import subprocess

pacotes = [
    "duckdb>=1.2",
    "pandas>=2.0",
    "pyarrow>=15",
    "folium>=0.20",
    "branca>=0.8",
    "geopandas>=1.0",
    "geobr>=1.0.0",
    "ipywidgets>=8",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *pacotes,
    ]
)

print("✅ Dependências instaladas.")

In [ ]:
# ============================================================
# ☁️ GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print("✅ Google Drive montado.")

## 1️⃣ Configuração de caminhos

O notebook trabalha em uma pasta independente:

```text
Suprimentos de Fundos - CPGF/
├── dados_consolidado/
│   └── CPGF_201301_a_202607.csv
├── dados_auxiliares/
│   └── siafi_dados_ug_2025.csv
└── Analise_Geografica_CPGF/
    ├── 00_controle/
    ├── 01_dimensoes/
    ├── 02_enriquecido/
    ├── 03_agregados/
    └── 04_mapas/
```

Se o arquivo SIAFI estiver em outra pasta, basta alterar `SIAFI_CSV`.

In [ ]:
# ============================================================
# 📁 CAMINHOS
# ============================================================

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Suprimentos de Fundos - CPGF"
)

CPGF_CSV = (
    BASE_DIR
    / "dados_consolidado"
    / "CPGF_201301_a_202607.csv"
)

SIAFI_CSV = (
    BASE_DIR
    / "dados_auxiliares"
    / "siafi_dados_ug_2025.csv"
)

RESULT_DIR = (
    BASE_DIR
    / "Analise_Geografica_CPGF_v1_1"
)

CONTROLE_DIR = RESULT_DIR / "00_controle"
DIM_DIR = RESULT_DIR / "01_dimensoes"
ENRIQUECIDO_DIR = RESULT_DIR / "02_enriquecido"
AGG_DIR = RESULT_DIR / "03_agregados"
MAPAS_DIR = RESULT_DIR / "04_mapas"

TMP_DIR = Path(
    "/content/cpgf_geo_tmp"
)

for pasta in [
    RESULT_DIR,
    CONTROLE_DIR,
    DIM_DIR,
    ENRIQUECIDO_DIR,
    AGG_DIR,
    MAPAS_DIR,
    TMP_DIR,
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

VERSAO_ENRIQUECIMENTO = "1.1.0"
VERSAO_CADASTRO_UF = "2025"

EXIGIR_COBERTURA_100 = True
GERAR_PARQUET_ENRIQUECIDO = True
ANO_GEOMETRIA_IBGE = 2025

print("📥 CPGF:", CPGF_CSV)
print("📥 SIAFI:", SIAFI_CSV)
print("📤 Resultados:", RESULT_DIR)
print("🗺️ Versão:", VERSAO_ENRIQUECIMENTO)

In [ ]:
# ============================================================
# 📚 IMPORTS E CONSTANTES
# ============================================================

import csv
import hashlib
import json
import math
import re
import time
from datetime import datetime

import duckdb
import folium
import geopandas as gpd
import numpy as np
import pandas as pd

from branca.colormap import linear
from folium.plugins import Fullscreen
from IPython.display import display
from tqdm.auto import tqdm

UFS_BRASIL = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PR",
    "PB", "PA", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SE", "SP", "TO",
]

UF_DOMINIO_SIAFI = UFS_BRASIL + ["EX"]

CODIGO_COMPRA_NACIONAL = "COMPRA A/V - R$ - APRES"
CODIGO_COMPRA_INTERNACIONAL = "COMPRA A/V - INT$ - APRES"
CODIGO_COMPRA_PARCELADA = "CPP LOJISTA TRF P/FATURA - REAL"

COMPRAS_OBSERVAVEIS = [
    CODIGO_COMPRA_NACIONAL,
    CODIGO_COMPRA_INTERNACIONAL,
    CODIGO_COMPRA_PARCELADA,
]

SAQUES_OBSERVAVEIS = [
    "SAQUE CASH/ATM BB",
    "SAQUE - INT$ - APRES",
    "SAQUE MANUAL - CARTOES BB NA AGENCIA",
    "SAQUE - R$ - APRES",
]

CODIGOS_AJUSTE_CONTESTACAO = [
    "COMP A/V-SOL DISP C/CLI-R$ ANT VENC",
    "COMP A/V-SOL DISP C/CLI-R$ APOS VENC",
    "SAQUE BB B24HORAS-SOL C/CLIENTE",
    "VOUCHER - R$ - REVRS REAPR",
]

COMPLEMENTOS_MANUAIS = {
    "511328": {
        "UF": "SP",
        "TITULO_UG": "Gerência-Executiva São Paulo - Norte (GEXSPN)",
    },
    "511341": {
        "UF": "SP",
        "TITULO_UG": "GERENCIA EXECUTIVA SAO PAULO-LESTE",
    },
    "510356": {
        "UF": "ES",
        "TITULO_UG": "UNID.TEC.DE REAB.PROFISSIONAL VITORIA",
    },
    "110703": {
        "UF": "DF",
        "TITULO_UG": "SUBSECRETARIA DE PLANEJAMENTO E GESTAO",
    },
    "110745": {
        "UF": "DF",
        "TITULO_UG": "SECRETARIA ESPECIAL DE AQUICULTURA E PESCA/PR",
    },
}

print("✅ Constantes carregadas.")

In [ ]:
# ============================================================
# ✅ VALIDAÇÃO DOS ARQUIVOS
# ============================================================

assert CPGF_CSV.exists(), (
    f"Arquivo CPGF não encontrado: {CPGF_CSV}"
)

assert SIAFI_CSV.exists(), (
    f"Arquivo SIAFI não encontrado: {SIAFI_CSV}"
)

print("✅ Arquivos encontrados.")
print("CPGF:", f"{CPGF_CSV.stat().st_size / 1024**2:.1f} MB")
print("SIAFI:", f"{SIAFI_CSV.stat().st_size / 1024**2:.1f} MB")

## 2️⃣ Chave canônica da Unidade Gestora

A chave do relacionamento será sempre:

```text
UG_ID = texto de 6 dígitos
```

Exemplos:

```text
133    → 000133
313    → 000313
110161 → 110161
```

No pipeline analítico, código identificador é tratado como **string**, não como quantidade numérica.

In [ ]:
# ============================================================
# 🧭 PARSER ROBUSTO SIAFI — UG + UF
# ============================================================

UF_REGEX = "|".join(
    UF_DOMINIO_SIAFI
)

PAT_UG_UF = re.compile(
    r'^"(?P<UG>\d{1,6})",'
    r'.*?,'
    r'"(?P<UF>' + UF_REGEX + r')",'
    r'"(?P<COD_ORGAO>\d{1,5})",'
)

def titulo_best_effort(linha):
    try:
        row = next(
            csv.reader(
                [linha]
            )
        )
        if len(row) >= 2:
            return row[1].strip()
    except Exception:
        pass
    return None

def carregar_dim_siafi(caminho):
    registros = []
    falhas = []

    with open(
        caminho,
        "r",
        encoding="utf-8-sig"
    ) as f:
        _ = next(f)

        for linha_numero, linha in enumerate(
            f,
            start=2
        ):
            m = PAT_UG_UF.search(
                linha
            )

            if not m:
                falhas.append({
                    "LINHA": linha_numero,
                    "TRECHO": linha[:250],
                })
                continue

            registros.append({
                "UG_ID": m.group("UG").zfill(6),
                "UF": m.group("UF"),
                "CODIGO_ORGAO_SIAFI": m.group("COD_ORGAO").zfill(5),
                "TITULO_UG_SIAFI": titulo_best_effort(linha),
                "FONTE_UF": "SIAFI_DADOS_UG_2025",
                "LINHA_ORIGEM_SIAFI": linha_numero,
            })

    return (
        pd.DataFrame(registros),
        pd.DataFrame(falhas),
    )

dim_siafi_df, falhas_siafi_df = carregar_dim_siafi(
    SIAFI_CSV
)

display(
    dim_siafi_df.head()
)

print("Linhas SIAFI parseadas:", f"{len(dim_siafi_df):,}")
print("Falhas de UG/UF:", len(falhas_siafi_df))

In [ ]:
# ============================================================
# 🔍 QUALIDADE DA DIMENSÃO SIAFI
# ============================================================

assert (
    dim_siafi_df["UG_ID"]
    .str.fullmatch(r"\d{6}")
    .all()
)

assert (
    ~dim_siafi_df["UG_ID"]
    .duplicated()
    .any()
)

assert set(
    dim_siafi_df["UF"]
    .dropna()
    .unique()
).issubset(
    set(UF_DOMINIO_SIAFI)
)

assert len(
    falhas_siafi_df
) == 0

# Metadados explícitos da origem da UF.
dim_siafi_df["TIPO_FONTE_UF"] = (
    "CADASTRO_SIAFI"
)

dim_siafi_df["VERSAO_FONTE_UF"] = (
    VERSAO_CADASTRO_UF
)

print("✅ UG SIAFI única e com seis dígitos.")
print("✅ UF dentro do domínio esperado.")
print("✅ Parser geográfico cobriu todas as linhas do SIAFI.")
print("UGs no cadastro:", f"{dim_siafi_df['UG_ID'].nunique():,}")
print(
    "UGs no exterior (EX):",
    int((dim_siafi_df["UF"] == "EX").sum())
)

## 3️⃣ Complementos manuais

As cinco UGs abaixo não estavam presentes no arquivo SIAFI fornecido, mas tiveram UF informada para esta POC.

Esses registros **não são apresentados como se viessem do SIAFI**.

A coluna `FONTE_UF` preservará:

```text
COMPLEMENTO_MANUAL_2026_08_13
```

In [ ]:
# ============================================================
# 🧩 APLICAR COMPLEMENTOS MANUAIS
# ============================================================

linhas_manual = []

for ug, info in COMPLEMENTOS_MANUAIS.items():
    linhas_manual.append({
        "UG_ID": str(ug).zfill(6),
        "UF": info["UF"],
        "CODIGO_ORGAO_SIAFI": None,
        "TITULO_UG_SIAFI": info["TITULO_UG"],
        "FONTE_UF": "COMPLEMENTO_MANUAL_2026_08_13",
        "TIPO_FONTE_UF": "COMPLEMENTO_MANUAL",
        "VERSAO_FONTE_UF": "2026-08-13",
        "LINHA_ORIGEM_SIAFI": None,
    })

dim_manual_df = pd.DataFrame(
    linhas_manual
)

sobreposicao_manual = (
    set(dim_manual_df["UG_ID"])
    &
    set(dim_siafi_df["UG_ID"])
)

assert not sobreposicao_manual, (
    "Complemento manual colide com UG já existente no SIAFI: "
    f"{sorted(sobreposicao_manual)}"
)

dim_ug_df = pd.concat(
    [
        dim_siafi_df,
        dim_manual_df,
    ],
    ignore_index=True
)

assert ~dim_ug_df["UG_ID"].duplicated().any()
assert dim_ug_df["UG_ID"].str.fullmatch(r"\d{6}").all()

display(
    dim_manual_df[
        [
            "UG_ID",
            "TITULO_UG_SIAFI",
            "UF",
            "TIPO_FONTE_UF",
            "VERSAO_FONTE_UF",
        ]
    ]
)

print("✅ Dimensão final criada.")
print("UGs finais:", f"{dim_ug_df['UG_ID'].nunique():,}")

In [ ]:
# ============================================================
# 💾 EXPORTAR DIMENSÃO UG
# ============================================================

DIM_UG_PARQUET = DIM_DIR / "dim_ug_geografica.parquet"
DIM_UG_CSV = DIM_DIR / "dim_ug_geografica.csv"

dim_ug_df.to_parquet(
    DIM_UG_PARQUET,
    index=False
)

dim_ug_df.to_csv(
    DIM_UG_CSV,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Dimensão exportada:")
print(DIM_UG_PARQUET)

## 4️⃣ Leitura do CPGF com DuckDB

O CSV consolidado permanece **imutável**.

O notebook cria:

```text
raw_cpgf
    ↓
normalização da UG
    ↓
LEFT JOIN dim_ug_geografica
    ↓
cpgf_enriquecido_uf.parquet
```

### Métrica territorial principal

O mapa principal utiliza:

> **valor transacionado positivo, excluindo códigos de ajuste/contestação**

A definição preserva transações sob sigilo e evita chamar de “compras” registros cujo tipo não está publicamente observável.

In [ ]:
# ============================================================
# 🦆 DUCKDB
# ============================================================

DB_PATH = TMP_DIR / "cpgf_geo.duckdb"

if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(
    str(DB_PATH)
)

con.execute(
    "PRAGMA threads=8"
)

con.execute(
    "PRAGMA memory_limit='8GB'"
)

cpgf_path_sql = str(
    CPGF_CSV
).replace(
    "'",
    "''"
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW raw_cpgf AS

    SELECT *
    FROM read_csv(
        '{cpgf_path_sql}',
        delim=';',
        header=true,
        all_varchar=true
    )
    """
)

print("✅ View raw_cpgf criada.")

In [ ]:
# ============================================================
# 🔎 PERFIL DA BASE CPGF
# ============================================================

perfil_cpgf_df = con.execute(
    """
    SELECT
        COUNT(*) AS N_REGISTROS,

        COUNT(
            DISTINCT LPAD(
                TRIM(
                    "CÓDIGO UNIDADE GESTORA"
                ),
                6,
                '0'
            )
        ) AS N_UG,

        MIN(
            TRY_STRPTIME(
                NULLIF(
                    TRIM(
                        "DATA TRANSAÇÃO"
                    ),
                    ''
                ),
                '%d/%m/%Y'
            )::DATE
        ) AS DATA_MIN,

        MAX(
            TRY_STRPTIME(
                NULLIF(
                    TRIM(
                        "DATA TRANSAÇÃO"
                    ),
                    ''
                ),
                '%d/%m/%Y'
            )::DATE
        ) AS DATA_MAX

    FROM raw_cpgf
    """
).df()

display(
    perfil_cpgf_df
)

print("✅ Perfil CPGF carregado.")

In [ ]:
# ============================================================
# 🔗 REGISTRAR DIMENSÃO NO DUCKDB
# ============================================================

con.register(
    "dim_ug_pandas",
    dim_ug_df
)

con.execute(
    """
    CREATE OR REPLACE TABLE dim_ug AS

    SELECT
        CAST(UG_ID AS VARCHAR) AS UG_ID,
        CAST(UF AS VARCHAR) AS UF,
        CAST(TITULO_UG_SIAFI AS VARCHAR) AS TITULO_UG_SIAFI,
        CAST(FONTE_UF AS VARCHAR) AS FONTE_UF,
        CAST(TIPO_FONTE_UF AS VARCHAR) AS TIPO_FONTE_UF,
        CAST(VERSAO_FONTE_UF AS VARCHAR) AS VERSAO_FONTE_UF,
        CAST(CODIGO_ORGAO_SIAFI AS VARCHAR) AS CODIGO_ORGAO_SIAFI

    FROM dim_ug_pandas
    """
)

print("✅ dim_ug registrada no DuckDB.")

## 5️⃣ Cobertura antes e depois dos complementos

Na base atual, espera-se:

- `2.153` UGs distintas no CPGF;
- `2.148` localizadas diretamente no arquivo SIAFI;
- `5` complementadas manualmente;
- cobertura final de `100%`.

Se uma atualização futura introduzir nova UG sem UF, o notebook exporta a pendência e, em modo estrito, interrompe o processamento.

In [ ]:
# ============================================================
# 📐 COBERTURA GEOGRÁFICA
# ============================================================

dim_siafi_join = dim_siafi_df[
    ["UG_ID", "UF"]
].copy()

con.register(
    "dim_siafi_sem_manual",
    dim_siafi_join
)

cobertura_df = con.execute(
    """
    WITH base AS (
        SELECT
            LPAD(
                TRIM(
                    "CÓDIGO UNIDADE GESTORA"
                ),
                6,
                '0'
            ) AS UG_ID
        FROM raw_cpgf
    ),

    por_ug AS (
        SELECT DISTINCT UG_ID
        FROM base
    )

    SELECT
        (
            SELECT COUNT(*)
            FROM por_ug
        ) AS N_UG_CPGF,

        (
            SELECT COUNT(*)
            FROM por_ug p
            JOIN dim_siafi_sem_manual s
                USING (UG_ID)
        ) AS N_UG_MATCH_SIAFI,

        (
            SELECT COUNT(*)
            FROM por_ug p
            JOIN dim_ug d
                USING (UG_ID)
        ) AS N_UG_MATCH_FINAL,

        (
            SELECT COUNT(*)
            FROM base b
            JOIN dim_ug d
                USING (UG_ID)
        ) AS N_REGISTROS_COM_UF,

        (
            SELECT COUNT(*)
            FROM base
        ) AS N_REGISTROS_TOTAL
    """
).df()

cobertura_df["COBERTURA_UG_PCT"] = (
    cobertura_df["N_UG_MATCH_FINAL"]
    /
    cobertura_df["N_UG_CPGF"]
    * 100
)

cobertura_df["COBERTURA_REGISTROS_PCT"] = (
    cobertura_df["N_REGISTROS_COM_UF"]
    /
    cobertura_df["N_REGISTROS_TOTAL"]
    * 100
)

display(
    cobertura_df
)

ugs_sem_uf_df = con.execute(
    """
    WITH ugs AS (
        SELECT DISTINCT
            LPAD(
                TRIM(
                    "CÓDIGO UNIDADE GESTORA"
                ),
                6,
                '0'
            ) AS UG_ID,

            "NOME UNIDADE GESTORA" AS NOME_UG_CPGF

        FROM raw_cpgf
    )

    SELECT u.*
    FROM ugs u

    LEFT JOIN dim_ug d
        USING (UG_ID)

    WHERE d.UF IS NULL

    ORDER BY UG_ID
    """
).df()

ugs_sem_uf_df.to_csv(
    CONTROLE_DIR / "ugs_sem_uf.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

if len(ugs_sem_uf_df):
    print("⚠️ UGs ainda sem UF:")
    display(ugs_sem_uf_df)
else:
    print("✅ Todas as UGs do CPGF possuem UF.")

if EXIGIR_COBERTURA_100:
    assert len(ugs_sem_uf_df) == 0, (
        "Cobertura inferior a 100%. "
        "Atualize a dimensão antes de prosseguir."
    )

## 6️⃣ Construção da camada enriquecida — V1.1

O Parquet enriquecido passa a carregar **duas referências temporais independentes**:

```text
ANO_TRANSACAO
ANO_EXTRATO_REF
```

`ANO_EXTRATO_REF` utiliza `ANO EXTRATO` e, apenas como fallback técnico para o mesmo conceito de competência, os quatro primeiros dígitos de `COMPETENCIA_ARQUIVO`.

Não existe fallback de `ANO_TRANSACAO` para `ANO_EXTRATO_REF`.

### Novos campos centrais

```text
UG_ID
UF_UG
FONTE_UF
TIPO_FONTE_UF
VERSAO_FONTE_UF

DATA_DT
ANO_TRANSACAO
ANO_EXTRATO_REF
EH_DATA_TRANSACAO_OBSERVAVEL

VALOR_NUM
VALOR_CENTAVOS
EH_AJUSTE_CONTESTACAO
EH_OPERACAO_POSITIVA_NAO_AJUSTE
EH_COMPRA_OBSERVAVEL
EH_SAQUE_OBSERVAVEL
EH_SIGILOSO
```

A ausência da data da transação passa a ser tratada como **característica de observabilidade**, e não como motivo para retirar o registro da visão territorial por extrato.

In [ ]:
# ============================================================
# 🧱 PREPARAÇÃO + ENRIQUECIMENTO — V1.1
# ============================================================

compras_sql = ", ".join(
    "'" + x.replace("'", "''") + "'"
    for x in COMPRAS_OBSERVAVEIS
)

saques_sql = ", ".join(
    "'" + x.replace("'", "''") + "'"
    for x in SAQUES_OBSERVAVEIS
)

ajustes_sql = ", ".join(
    "'" + x.replace("'", "''") + "'"
    for x in CODIGOS_AJUSTE_CONTESTACAO
)

ENRIQUECIDO_PARQUET = (
    ENRIQUECIDO_DIR
    / "cpgf_enriquecido_uf_v1_1.parquet"
)

enriquecido_path_sql = str(
    ENRIQUECIDO_PARQUET
).replace(
    "'",
    "''"
)

sql_enriquecimento = f"""
COPY (
    WITH preparado AS (
        SELECT
            r.*,

            LPAD(
                TRIM(
                    r."CÓDIGO UNIDADE GESTORA"
                ),
                6,
                '0'
            ) AS UG_ID,

            TRY_STRPTIME(
                NULLIF(
                    TRIM(
                        r."DATA TRANSAÇÃO"
                    ),
                    ''
                ),
                '%d/%m/%Y'
            )::DATE AS DATA_DT,

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        TRIM(
                            r."ANO EXTRATO"
                        ),
                        ''
                    )
                    AS INTEGER
                ),

                TRY_CAST(
                    SUBSTR(
                        TRIM(
                            COALESCE(
                                r."COMPETENCIA_ARQUIVO",
                                ''
                            )
                        ),
                        1,
                        4
                    )
                    AS INTEGER
                )
            ) AS ANO_EXTRATO_REF,

            TRY_CAST(
                CASE
                    WHEN CONTAINS(
                        REPLACE(
                            REPLACE(
                                TRIM(
                                    r."VALOR TRANSAÇÃO"
                                ),
                                'R$',
                                ''
                            ),
                            ' ',
                            ''
                        ),
                        ','
                    )
                    THEN REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    TRIM(
                                        r."VALOR TRANSAÇÃO"
                                    ),
                                    'R$',
                                    ''
                                ),
                                ' ',
                                ''
                            ),
                            '.',
                            ''
                        ),
                        ',',
                        '.'
                    )
                    ELSE REPLACE(
                        REPLACE(
                            TRIM(
                                r."VALOR TRANSAÇÃO"
                            ),
                            'R$',
                            ''
                        ),
                        ' ',
                        ''
                    )
                END
                AS DECIMAL(18,2)
            ) AS VALOR_NUM

        FROM raw_cpgf r
    ),

    flags AS (
        SELECT
            p.*,

            EXTRACT(
                YEAR FROM DATA_DT
            )::INTEGER AS ANO_TRANSACAO,

            (
                DATA_DT IS NOT NULL
            ) AS EH_DATA_TRANSACAO_OBSERVAVEL,

            CAST(
                ROUND(
                    VALOR_NUM * 100,
                    0
                )
                AS BIGINT
            ) AS VALOR_CENTAVOS,

            (
                TRIM(
                    COALESCE(
                        p."TRANSAÇÃO",
                        ''
                    )
                )
                IN ({ajustes_sql})
            ) AS EH_AJUSTE_CONTESTACAO,

            (
                TRIM(
                    COALESCE(
                        p."TRANSAÇÃO",
                        ''
                    )
                )
                IN ({compras_sql})
            ) AS EH_COMPRA_CODIGO,

            (
                TRIM(
                    COALESCE(
                        p."TRANSAÇÃO",
                        ''
                    )
                )
                IN ({saques_sql})
            ) AS EH_SAQUE_CODIGO,

            (
                LOWER(
                    COALESCE(
                        p."TRANSAÇÃO",
                        ''
                    )
                ) LIKE '%sigilo%'

                OR LOWER(
                    COALESCE(
                        p."NOME PORTADOR",
                        ''
                    )
                ) LIKE '%sigilo%'

                OR LOWER(
                    COALESCE(
                        p."NOME FAVORECIDO",
                        ''
                    )
                ) LIKE '%sigilo%'
            ) AS EH_SIGILO_BASE

        FROM preparado p
    )

    SELECT
        f.*,

        d.UF AS UF_UG,
        d.TITULO_UG_SIAFI,
        d.FONTE_UF,
        d.TIPO_FONTE_UF,
        d.VERSAO_FONTE_UF,
        d.CODIGO_ORGAO_SIAFI,

        (
            VALOR_NUM > 0
            AND NOT EH_AJUSTE_CONTESTACAO
        ) AS EH_OPERACAO_POSITIVA_NAO_AJUSTE,

        (
            VALOR_NUM > 0
            AND NOT EH_AJUSTE_CONTESTACAO
            AND EH_COMPRA_CODIGO
        ) AS EH_COMPRA_OBSERVAVEL,

        (
            VALOR_NUM > 0
            AND NOT EH_AJUSTE_CONTESTACAO
            AND EH_SAQUE_CODIGO
        ) AS EH_SAQUE_OBSERVAVEL,

        (
            VALOR_NUM > 0
            AND NOT EH_AJUSTE_CONTESTACAO
            AND EH_SIGILO_BASE
        ) AS EH_SIGILOSO

    FROM flags f

    LEFT JOIN dim_ug d
        USING (UG_ID)
)
TO '{enriquecido_path_sql}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

inicio = time.time()

if GERAR_PARQUET_ENRIQUECIDO:
    con.execute(
        sql_enriquecimento
    )

    print(
        "✅ Parquet enriquecido V1.1 criado em "
        f"{time.time() - inicio:.1f}s."
    )
else:
    print("⏭️ Geração do Parquet desativada.")

In [ ]:
# ============================================================
# 🔍 VALIDAÇÃO DO ENRIQUECIMENTO — V1.1
# ============================================================

assert ENRIQUECIDO_PARQUET.exists()

enriquecido_path_sql = str(
    ENRIQUECIDO_PARQUET
).replace(
    "'",
    "''"
)

qualidade_enriquecido_df = con.execute(
    f"""
    SELECT
        COUNT(*) AS N_REGISTROS,

        COUNT_IF(
            UF_UG IS NULL
        ) AS N_SEM_UF,

        COUNT(
            DISTINCT UG_ID
        ) AS N_UG,

        COUNT(
            DISTINCT CASE
                WHEN UF_UG IS NOT NULL
                THEN UG_ID
            END
        ) AS N_UG_COM_UF,

        COUNT_IF(
            FONTE_UF
            = 'COMPLEMENTO_MANUAL_2026_08_13'
        ) AS N_REGISTROS_COMPLEMENTO_MANUAL,

        COUNT_IF(
            EH_DATA_TRANSACAO_OBSERVAVEL
        ) AS N_COM_DATA_TRANSACAO,

        COUNT_IF(
            NOT EH_DATA_TRANSACAO_OBSERVAVEL
        ) AS N_SEM_DATA_TRANSACAO,

        COUNT_IF(
            ANO_EXTRATO_REF IS NOT NULL
        ) AS N_COM_ANO_EXTRATO,

        SUM(
            CASE
                WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS VALOR_POSITIVO_SEM_AJUSTE,

        SUM(
            CASE
                WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                 AND EH_DATA_TRANSACAO_OBSERVAVEL
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS VALOR_COM_DATA_TRANSACAO,

        SUM(
            CASE
                WHEN EH_SIGILOSO
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS VALOR_SIGILOSO_TOTAL

    FROM read_parquet(
        '{enriquecido_path_sql}'
    )
    """
).df()

display(
    qualidade_enriquecido_df
)

assert int(
    qualidade_enriquecido_df[
        "N_SEM_UF"
    ].iloc[0]
) == 0

print("✅ Cobertura geográfica final = 100%.")
print(
    "ℹ️ Cobertura temporal por DATA TRANSAÇÃO é "
    "medida separadamente da cobertura geográfica."
)

## 7️⃣ Duas tabelas territoriais `UF × ano`

A V1.1 separa formalmente dois universos.

### A. `agg_cpgf_uf_ano_transacao`

Referência:

```text
ANO = ANO_TRANSACAO
```

Métricas principais:

- `VALOR_TRANSACIONADO_OBSERVAVEL`;
- `N_TRANSACOES_OBSERVAVEIS`;
- `N_UG_COM_MOVIMENTACAO`;
- `VALOR_MEDIO_POR_UG`;
- `PARTICIPACAO_NACIONAL_PCT`;
- compras e saques observáveis.

A palavra **observável** é deliberada: registros sem `DATA TRANSAÇÃO` não podem ser atribuídos a um ano de transação.

### B. `agg_cpgf_uf_ano_extrato`

Referência:

```text
ANO = ANO_EXTRATO_REF
```

Métricas principais:

- `VALOR_TOTAL_REGISTRADO`;
- `N_REGISTROS`;
- `N_UG_COM_MOVIMENTACAO`;
- `VALOR_SIGILOSO`;
- `PCT_SIGILO_VALOR`;
- `VALOR_COM_DATA_TRANSACAO_OBSERVAVEL`;
- `TAXA_OBSERVABILIDADE_VALOR`;
- `TAXA_OBSERVABILIDADE_REGISTROS`.

Essa segunda visão é a apropriada para falar em **cobertura integral do extrato e sigilo**, sem fingir conhecer a data efetiva das operações protegidas.

In [ ]:
# ============================================================
# 📊 AGREGADOS V1.1 — TRANSAÇÃO E EXTRATO
# ============================================================

ufs_sql = ", ".join(
    "'" + uf + "'"
    for uf in UFS_BRASIL
)

# ------------------------------------------------------------
# A) VISÃO POR ANO DA TRANSAÇÃO
# ------------------------------------------------------------

agg_transacao_df = con.execute(
    f"""
    WITH agg AS (
        SELECT
            ANO_TRANSACAO AS ANO,
            UF_UG AS UF,

            SUM(
                CASE
                    WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_TRANSACIONADO_OBSERVAVEL,

            COUNT_IF(
                EH_OPERACAO_POSITIVA_NAO_AJUSTE
            ) AS N_TRANSACOES_OBSERVAVEIS,

            COUNT(
                DISTINCT CASE
                    WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                    THEN UG_ID
                END
            ) AS N_UG_COM_MOVIMENTACAO,

            SUM(
                CASE
                    WHEN EH_COMPRA_OBSERVAVEL
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_COMPRAS_OBSERVAVEIS,

            SUM(
                CASE
                    WHEN EH_SAQUE_OBSERVAVEL
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_SAQUES_OBSERVAVEIS

        FROM read_parquet(
            '{enriquecido_path_sql}'
        )

        WHERE ANO_TRANSACAO IS NOT NULL
          AND UF_UG IN ({ufs_sql})

        GROUP BY 1, 2
    )

    SELECT
        *,

        CASE
            WHEN N_UG_COM_MOVIMENTACAO > 0
            THEN (
                VALOR_TRANSACIONADO_OBSERVAVEL
                / N_UG_COM_MOVIMENTACAO
            )
            ELSE NULL
        END AS VALOR_MEDIO_POR_UG,

        (
            VALOR_TRANSACIONADO_OBSERVAVEL
            /
            NULLIF(
                SUM(
                    VALOR_TRANSACIONADO_OBSERVAVEL
                ) OVER (
                    PARTITION BY ANO
                ),
                0
            )
            * 100
        ) AS PARTICIPACAO_NACIONAL_PCT,

        CASE
            WHEN ANO BETWEEN 2013 AND 2025
            THEN 'EXERCICIO_COMPLETO'
            ELSE 'PERIODO_PARCIAL'
        END AS STATUS_PERIODO

    FROM agg
    ORDER BY ANO, UF
    """
).df()


# ------------------------------------------------------------
# B) VISÃO POR ANO DO EXTRATO
# ------------------------------------------------------------

agg_extrato_df = con.execute(
    f"""
    WITH agg AS (
        SELECT
            ANO_EXTRATO_REF AS ANO,
            UF_UG AS UF,

            SUM(
                CASE
                    WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_TOTAL_REGISTRADO,

            COUNT_IF(
                EH_OPERACAO_POSITIVA_NAO_AJUSTE
            ) AS N_REGISTROS,

            COUNT(
                DISTINCT CASE
                    WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                    THEN UG_ID
                END
            ) AS N_UG_COM_MOVIMENTACAO,

            SUM(
                CASE
                    WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                     AND EH_DATA_TRANSACAO_OBSERVAVEL
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_COM_DATA_TRANSACAO_OBSERVAVEL,

            COUNT_IF(
                EH_OPERACAO_POSITIVA_NAO_AJUSTE
                AND EH_DATA_TRANSACAO_OBSERVAVEL
            ) AS N_REGISTROS_COM_DATA_TRANSACAO,

            SUM(
                CASE
                    WHEN EH_SIGILOSO
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_SIGILOSO,

            COUNT_IF(
                EH_SIGILOSO
            ) AS N_REGISTROS_SIGILOSOS,

            SUM(
                CASE
                    WHEN EH_COMPRA_OBSERVAVEL
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_COMPRAS_OBSERVAVEIS,

            SUM(
                CASE
                    WHEN EH_SAQUE_OBSERVAVEL
                    THEN VALOR_NUM
                    ELSE 0
                END
            )::DOUBLE AS VALOR_SAQUES_OBSERVAVEIS

        FROM read_parquet(
            '{enriquecido_path_sql}'
        )

        WHERE ANO_EXTRATO_REF IS NOT NULL
          AND UF_UG IN ({ufs_sql})

        GROUP BY 1, 2
    )

    SELECT
        *,

        CASE
            WHEN N_UG_COM_MOVIMENTACAO > 0
            THEN (
                VALOR_TOTAL_REGISTRADO
                / N_UG_COM_MOVIMENTACAO
            )
            ELSE NULL
        END AS VALOR_MEDIO_POR_UG,

        (
            VALOR_TOTAL_REGISTRADO
            /
            NULLIF(
                SUM(
                    VALOR_TOTAL_REGISTRADO
                ) OVER (
                    PARTITION BY ANO
                ),
                0
            )
            * 100
        ) AS PARTICIPACAO_NACIONAL_PCT,

        (
            VALOR_SIGILOSO
            /
            NULLIF(
                VALOR_TOTAL_REGISTRADO,
                0
            )
            * 100
        ) AS PCT_SIGILO_VALOR,

        (
            VALOR_COM_DATA_TRANSACAO_OBSERVAVEL
            /
            NULLIF(
                VALOR_TOTAL_REGISTRADO,
                0
            )
            * 100
        ) AS TAXA_OBSERVABILIDADE_VALOR,

        (
            N_REGISTROS_COM_DATA_TRANSACAO
            /
            NULLIF(
                N_REGISTROS,
                0
            )
            * 100
        ) AS TAXA_OBSERVABILIDADE_REGISTROS,

        CASE
            WHEN ANO BETWEEN 2013 AND 2025
            THEN 'EXERCICIO_COMPLETO'
            ELSE 'PERIODO_PARCIAL'
        END AS STATUS_PERIODO

    FROM agg
    ORDER BY ANO, UF
    """
).df()


display(
    agg_transacao_df.head(10)
)

display(
    agg_extrato_df.head(10)
)

print(
    "Linhas — transação:",
    len(agg_transacao_df)
)

print(
    "Linhas — extrato:",
    len(agg_extrato_df)
)

In [ ]:
# ============================================================
# 💾 CONTRATO SEMÂNTICO + EXPORTAÇÃO DOS AGREGADOS
# ============================================================

AGG_TRANSACAO_PARQUET = (
    AGG_DIR
    / "agg_cpgf_uf_ano_transacao.parquet"
)

AGG_TRANSACAO_CSV = (
    AGG_DIR
    / "agg_cpgf_uf_ano_transacao.csv"
)

AGG_EXTRATO_PARQUET = (
    AGG_DIR
    / "agg_cpgf_uf_ano_extrato.parquet"
)

AGG_EXTRATO_CSV = (
    AGG_DIR
    / "agg_cpgf_uf_ano_extrato.csv"
)

AGG_DASHBOARD_PARQUET = (
    AGG_DIR
    / "agg_cpgf_uf_ano_dashboard_long.parquet"
)

AGG_DASHBOARD_CSV = (
    AGG_DIR
    / "agg_cpgf_uf_ano_dashboard_long.csv"
)


# ------------------------------------------------------------
# CATÁLOGO DAS MÉTRICAS
# ------------------------------------------------------------

catalogo_metricas_df = pd.DataFrame([
    # TRANSACAO
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "VALOR_TRANSACIONADO_OBSERVAVEL",
        "ROTULO": "Valor transacionado observável",
        "UNIDADE": "BRL",
        "DESCRICAO": (
            "Valor positivo sem ajustes cuja DATA TRANSAÇÃO "
            "é observável e atribuível ao ano da operação."
        ),
        "METRICA_PRINCIPAL": True,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "PARTICIPACAO_NACIONAL_PCT",
        "ROTULO": "Participação nacional",
        "UNIDADE": "PERCENTUAL",
        "DESCRICAO": (
            "Participação da UF no valor observável nacional "
            "do mesmo ano da transação."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "VALOR_MEDIO_POR_UG",
        "ROTULO": "Valor médio por UG",
        "UNIDADE": "BRL",
        "DESCRICAO": (
            "Valor observável dividido pelas UGs com movimentação."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "N_TRANSACOES_OBSERVAVEIS",
        "ROTULO": "Quantidade de transações observáveis",
        "UNIDADE": "CONTAGEM",
        "DESCRICAO": (
            "Registros positivos sem ajustes com data de transação observável."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "N_UG_COM_MOVIMENTACAO",
        "ROTULO": "UGs com movimentação",
        "UNIDADE": "CONTAGEM",
        "DESCRICAO": "UGs com operação observável no ano da transação.",
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "VALOR_COMPRAS_OBSERVAVEIS",
        "ROTULO": "Compras observáveis",
        "UNIDADE": "BRL",
        "DESCRICAO": "Valor de compras com código observável.",
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "TRANSACAO",
        "METRICA": "VALOR_SAQUES_OBSERVAVEIS",
        "ROTULO": "Saques observáveis",
        "UNIDADE": "BRL",
        "DESCRICAO": "Valor de saques com código observável.",
        "METRICA_PRINCIPAL": False,
    },

    # EXTRATO
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "VALOR_TOTAL_REGISTRADO",
        "ROTULO": "Valor total registrado",
        "UNIDADE": "BRL",
        "DESCRICAO": (
            "Valor positivo sem ajustes atribuído ao ano do extrato; "
            "inclui registros cuja data de transação não é observável."
        ),
        "METRICA_PRINCIPAL": True,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "PARTICIPACAO_NACIONAL_PCT",
        "ROTULO": "Participação nacional",
        "UNIDADE": "PERCENTUAL",
        "DESCRICAO": (
            "Participação da UF no valor total registrado no ano do extrato."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "VALOR_MEDIO_POR_UG",
        "ROTULO": "Valor médio por UG",
        "UNIDADE": "BRL",
        "DESCRICAO": (
            "Valor total registrado dividido pelas UGs com movimentação."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "N_REGISTROS",
        "ROTULO": "Quantidade de registros",
        "UNIDADE": "CONTAGEM",
        "DESCRICAO": (
            "Registros positivos sem ajustes no ano do extrato."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "N_UG_COM_MOVIMENTACAO",
        "ROTULO": "UGs com movimentação",
        "UNIDADE": "CONTAGEM",
        "DESCRICAO": "UGs com registro positivo no ano do extrato.",
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "VALOR_SIGILOSO",
        "ROTULO": "Valor sob sigilo",
        "UNIDADE": "BRL",
        "DESCRICAO": "Valor positivo identificado como sigiloso.",
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "PCT_SIGILO_VALOR",
        "ROTULO": "Percentual do valor sob sigilo",
        "UNIDADE": "PERCENTUAL",
        "DESCRICAO": (
            "Valor sigiloso dividido pelo valor total registrado da UF/ano."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "VALOR_COM_DATA_TRANSACAO_OBSERVAVEL",
        "ROTULO": "Valor com data da transação observável",
        "UNIDADE": "BRL",
        "DESCRICAO": (
            "Valor do mesmo extrato cuja DATA TRANSAÇÃO está publicamente observável."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "TAXA_OBSERVABILIDADE_VALOR",
        "ROTULO": "Observabilidade do valor",
        "UNIDADE": "PERCENTUAL",
        "DESCRICAO": (
            "Parcela do valor total registrado que possui DATA TRANSAÇÃO observável."
        ),
        "METRICA_PRINCIPAL": False,
    },
    {
        "REFERENCIA_TEMPORAL": "EXTRATO",
        "METRICA": "TAXA_OBSERVABILIDADE_REGISTROS",
        "ROTULO": "Observabilidade dos registros",
        "UNIDADE": "PERCENTUAL",
        "DESCRICAO": (
            "Parcela dos registros positivos que possui DATA TRANSAÇÃO observável."
        ),
        "METRICA_PRINCIPAL": False,
    },
])


def para_formato_longo(
    df,
    referencia
):
    catalogo = (
        catalogo_metricas_df[
            catalogo_metricas_df[
                "REFERENCIA_TEMPORAL"
            ]
            == referencia
        ]
        .copy()
    )

    partes = []

    for _, meta in catalogo.iterrows():
        metrica = meta["METRICA"]

        if metrica not in df.columns:
            continue

        tmp = df[
            [
                "ANO",
                "UF",
                "STATUS_PERIODO",
                metrica,
            ]
        ].copy()

        tmp = tmp.rename(
            columns={
                metrica: "VALOR_METRICA"
            }
        )

        tmp["REFERENCIA_TEMPORAL"] = referencia
        tmp["METRICA"] = metrica
        tmp["ROTULO_METRICA"] = meta["ROTULO"]
        tmp["UNIDADE"] = meta["UNIDADE"]

        partes.append(tmp)

    return pd.concat(
        partes,
        ignore_index=True
    )


agg_dashboard_long_df = pd.concat(
    [
        para_formato_longo(
            agg_transacao_df,
            "TRANSACAO"
        ),

        para_formato_longo(
            agg_extrato_df,
            "EXTRATO"
        ),
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# EXPORTAR
# ------------------------------------------------------------

agg_transacao_df.to_parquet(
    AGG_TRANSACAO_PARQUET,
    index=False
)

agg_transacao_df.to_csv(
    AGG_TRANSACAO_CSV,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

agg_extrato_df.to_parquet(
    AGG_EXTRATO_PARQUET,
    index=False
)

agg_extrato_df.to_csv(
    AGG_EXTRATO_CSV,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

agg_dashboard_long_df.to_parquet(
    AGG_DASHBOARD_PARQUET,
    index=False
)

agg_dashboard_long_df.to_csv(
    AGG_DASHBOARD_CSV,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

catalogo_metricas_df.to_csv(
    CONTROLE_DIR
    / "catalogo_metricas_territoriais.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Agregados V1.1 exportados.")
print("A:", AGG_TRANSACAO_PARQUET)
print("B:", AGG_EXTRATO_PARQUET)
print("Dashboard:", AGG_DASHBOARD_PARQUET)

In [ ]:
# ============================================================
# 🧪 CHECKS DE INTEGRIDADE E OBSERVABILIDADE
# ============================================================

# ------------------------------------------------------------
# 1. Participação nacional fecha em 100% por ano.
# ------------------------------------------------------------

share_transacao_df = (
    agg_transacao_df
    .groupby(
        "ANO",
        as_index=False
    )[
        "PARTICIPACAO_NACIONAL_PCT"
    ]
    .sum()
)

share_extrato_df = (
    agg_extrato_df
    .groupby(
        "ANO",
        as_index=False
    )[
        "PARTICIPACAO_NACIONAL_PCT"
    ]
    .sum()
)

assert np.allclose(
    share_transacao_df[
        "PARTICIPACAO_NACIONAL_PCT"
    ],
    100.0,
    atol=1e-8
)

assert np.allclose(
    share_extrato_df[
        "PARTICIPACAO_NACIONAL_PCT"
    ],
    100.0,
    atol=1e-8
)


# ------------------------------------------------------------
# 2. Totais agregados = fonte enriquecida no mesmo universo.
# ------------------------------------------------------------

totais_fonte_df = con.execute(
    f"""
    SELECT
        SUM(
            CASE
                WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                 AND ANO_TRANSACAO IS NOT NULL
                 AND UF_UG IN ({ufs_sql})
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS TOTAL_TRANSACAO_OBSERVAVEL,

        SUM(
            CASE
                WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                 AND ANO_EXTRATO_REF IS NOT NULL
                 AND UF_UG IN ({ufs_sql})
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS TOTAL_EXTRATO_REGISTRADO,

        SUM(
            CASE
                WHEN EH_SIGILOSO
                 AND ANO_EXTRATO_REF IS NOT NULL
                 AND UF_UG IN ({ufs_sql})
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS TOTAL_SIGILO_EXTRATO

    FROM read_parquet(
        '{enriquecido_path_sql}'
    )
    """
).df()

total_transacao_agg = float(
    agg_transacao_df[
        "VALOR_TRANSACIONADO_OBSERVAVEL"
    ].sum()
)

total_extrato_agg = float(
    agg_extrato_df[
        "VALOR_TOTAL_REGISTRADO"
    ].sum()
)

total_sigilo_agg = float(
    agg_extrato_df[
        "VALOR_SIGILOSO"
    ].sum()
)

assert math.isclose(
    total_transacao_agg,
    float(
        totais_fonte_df[
            "TOTAL_TRANSACAO_OBSERVAVEL"
        ].iloc[0]
    ),
    abs_tol=0.01
)

assert math.isclose(
    total_extrato_agg,
    float(
        totais_fonte_df[
            "TOTAL_EXTRATO_REGISTRADO"
        ].iloc[0]
    ),
    abs_tol=0.01
)

assert math.isclose(
    total_sigilo_agg,
    float(
        totais_fonte_df[
            "TOTAL_SIGILO_EXTRATO"
        ].iloc[0]
    ),
    abs_tol=0.01
)


# ------------------------------------------------------------
# 3. Resumo Brasil por ano do extrato.
# ------------------------------------------------------------

resumo_observabilidade_df = (
    agg_extrato_df
    .groupby(
        [
            "ANO",
            "STATUS_PERIODO",
        ],
        as_index=False
    )
    .agg(
        VALOR_TOTAL_REGISTRADO=(
            "VALOR_TOTAL_REGISTRADO",
            "sum"
        ),

        VALOR_COM_DATA_TRANSACAO_OBSERVAVEL=(
            "VALOR_COM_DATA_TRANSACAO_OBSERVAVEL",
            "sum"
        ),

        VALOR_SIGILOSO=(
            "VALOR_SIGILOSO",
            "sum"
        ),

        N_REGISTROS=(
            "N_REGISTROS",
            "sum"
        ),

        N_REGISTROS_COM_DATA_TRANSACAO=(
            "N_REGISTROS_COM_DATA_TRANSACAO",
            "sum"
        ),

        N_REGISTROS_SIGILOSOS=(
            "N_REGISTROS_SIGILOSOS",
            "sum"
        ),
    )
)

resumo_observabilidade_df[
    "TAXA_OBSERVABILIDADE_VALOR"
] = (
    resumo_observabilidade_df[
        "VALOR_COM_DATA_TRANSACAO_OBSERVAVEL"
    ]
    /
    resumo_observabilidade_df[
        "VALOR_TOTAL_REGISTRADO"
    ]
    * 100
)

resumo_observabilidade_df[
    "PCT_SIGILO_VALOR"
] = (
    resumo_observabilidade_df[
        "VALOR_SIGILOSO"
    ]
    /
    resumo_observabilidade_df[
        "VALOR_TOTAL_REGISTRADO"
    ]
    * 100
)

resumo_observabilidade_df[
    "TAXA_OBSERVABILIDADE_REGISTROS"
] = (
    resumo_observabilidade_df[
        "N_REGISTROS_COM_DATA_TRANSACAO"
    ]
    /
    resumo_observabilidade_df[
        "N_REGISTROS"
    ]
    * 100
)

resumo_observabilidade_df.to_csv(
    CONTROLE_DIR
    / "resumo_observabilidade_ano_extrato.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. Eventuais UGs no exterior não entram no choropleth.
# ------------------------------------------------------------

resumo_exterior_df = con.execute(
    f"""
    SELECT
        ANO_EXTRATO_REF AS ANO_EXTRATO,
        COUNT_IF(
            EH_OPERACAO_POSITIVA_NAO_AJUSTE
        ) AS N_REGISTROS,
        SUM(
            CASE
                WHEN EH_OPERACAO_POSITIVA_NAO_AJUSTE
                THEN VALOR_NUM
                ELSE 0
            END
        )::DOUBLE AS VALOR_TOTAL_REGISTRADO

    FROM read_parquet(
        '{enriquecido_path_sql}'
    )

    WHERE UF_UG = 'EX'
      AND ANO_EXTRATO_REF IS NOT NULL

    GROUP BY 1
    ORDER BY 1
    """
).df()

resumo_exterior_df.to_csv(
    CONTROLE_DIR
    / "resumo_ug_exterior.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)


print("✅ Integridade dos dois agregados validada.")
print(
    "Valor — visão por transação:",
    moeda_br(total_transacao_agg)
    if "moeda_br" in globals()
    else total_transacao_agg
)
print(
    "Valor — visão por extrato:",
    moeda_br(total_extrato_agg)
    if "moeda_br" in globals()
    else total_extrato_agg
)
print(
    "Valor sob sigilo — visão por extrato:",
    moeda_br(total_sigilo_agg)
    if "moeda_br" in globals()
    else total_sigilo_agg
)

display(
    resumo_observabilidade_df
)

if len(resumo_exterior_df):
    print(
        "ℹ️ Há registros de UGs classificadas como EX; "
        "eles são reportados separadamente e não entram no mapa das 27 UFs."
    )
    display(
        resumo_exterior_df
    )
else:
    print(
        "✅ Não há movimentação positiva de UGs EX no universo atual."
    )

## 8️⃣ Geometrias oficiais dos estados

O notebook usa `geobr` apenas para obter a geometria dos estados brasileiros.

Para visualização, utiliza-se geometria simplificada, adequada a mapas web.

A geometria não interfere nos valores; serve apenas para desenhar os polígonos.

In [ ]:
# ============================================================
# 🗺️ CARREGAR ESTADOS — GEOBR
# ============================================================

def carregar_estados_geobr(
    ano=2025
):
    erros = []

    try:
        from geobr import read_state

        estados = read_state(
            code_state="all",
            year=ano
        )

        if estados is not None:
            return estados

    except Exception as e:
        erros.append(
            f"read_state: {e}"
        )

    try:
        from geobr import to_geopandas

        estados = to_geopandas(
            f"states_{ano}"
        )

        if estados is not None:
            return estados

    except Exception as e:
        erros.append(
            f"to_geopandas: {e}"
        )

    raise RuntimeError(
        "Não foi possível carregar as geometrias. "
        + " | ".join(erros)
    )

estados_gdf = carregar_estados_geobr(
    ANO_GEOMETRIA_IBGE
)

if estados_gdf.crs is not None:
    estados_gdf = estados_gdf.to_crs(
        epsg=4326
    )

candidatas_uf = [
    c
    for c in estados_gdf.columns
    if c.lower()
    in {
        "abbrev_state",
        "sigla_uf",
        "uf",
        "abbrev",
    }
]

if not candidatas_uf:
    raise KeyError(
        "Não encontrei coluna de sigla UF no geobr. "
        f"Colunas: {list(estados_gdf.columns)}"
    )

COL_UF_GEO = candidatas_uf[0]

candidatas_nome = [
    c
    for c in estados_gdf.columns
    if c.lower()
    in {
        "name_state",
        "nome_uf",
        "name",
        "nome",
    }
]

COL_NOME_GEO = (
    candidatas_nome[0]
    if candidatas_nome
    else None
)

print("✅ Geometrias carregadas.")
print("Coluna UF:", COL_UF_GEO)

display(
    estados_gdf.head()
)

## 9️⃣ Contrato das métricas do mapa

A escolha da métrica passa a depender da referência temporal.

### Ano da transação

Disponível para dados cuja `DATA TRANSAÇÃO` é observável:

- valor transacionado observável;
- participação nacional;
- valor médio por UG;
- quantidade de transações observáveis;
- UGs com movimentação;
- compras observáveis;
- saques observáveis.

### Ano do extrato

Disponível para cobertura temporal integral do extrato:

- valor total registrado;
- participação nacional;
- valor médio por UG;
- quantidade de registros;
- UGs com movimentação;
- valor sob sigilo;
- percentual do valor sob sigilo;
- valor com data da transação observável;
- taxa de observabilidade do valor;
- taxa de observabilidade dos registros.

O mapa nunca apresenta `% sigilo` como métrica da visão `ANO_TRANSACAO`, porque essa população exclui justamente registros cuja data não está disponível.

In [ ]:
# ============================================================
# 🎨 METADADOS DAS MÉTRICAS — V1.1
# ============================================================

METRICAS_MAPA = {
    "TRANSACAO": {
        "VALOR_TRANSACIONADO_OBSERVAVEL": {
            "rotulo": "Valor transacionado observável",
            "legenda": "Valor com data da transação observável (R$)",
            "tipo": "moeda",
        },
        "PARTICIPACAO_NACIONAL_PCT": {
            "rotulo": "Participação nacional",
            "legenda": "Participação no valor observável nacional (%)",
            "tipo": "percentual",
        },
        "VALOR_MEDIO_POR_UG": {
            "rotulo": "Valor médio por UG",
            "legenda": "Valor observável médio por UG (R$)",
            "tipo": "moeda",
        },
        "N_TRANSACOES_OBSERVAVEIS": {
            "rotulo": "Quantidade de transações observáveis",
            "legenda": "Número de transações com data observável",
            "tipo": "inteiro",
        },
        "N_UG_COM_MOVIMENTACAO": {
            "rotulo": "UGs com movimentação",
            "legenda": "Número de UGs com movimentação observável",
            "tipo": "inteiro",
        },
        "VALOR_COMPRAS_OBSERVAVEIS": {
            "rotulo": "Compras observáveis",
            "legenda": "Valor de compras observáveis (R$)",
            "tipo": "moeda",
        },
        "VALOR_SAQUES_OBSERVAVEIS": {
            "rotulo": "Saques observáveis",
            "legenda": "Valor de saques observáveis (R$)",
            "tipo": "moeda",
        },
    },

    "EXTRATO": {
        "VALOR_TOTAL_REGISTRADO": {
            "rotulo": "Valor total registrado",
            "legenda": "Valor positivo sem ajustes no ano do extrato (R$)",
            "tipo": "moeda",
        },
        "PARTICIPACAO_NACIONAL_PCT": {
            "rotulo": "Participação nacional",
            "legenda": "Participação no valor registrado nacional (%)",
            "tipo": "percentual",
        },
        "VALOR_MEDIO_POR_UG": {
            "rotulo": "Valor médio por UG",
            "legenda": "Valor registrado médio por UG (R$)",
            "tipo": "moeda",
        },
        "N_REGISTROS": {
            "rotulo": "Quantidade de registros",
            "legenda": "Número de registros positivos sem ajustes",
            "tipo": "inteiro",
        },
        "N_UG_COM_MOVIMENTACAO": {
            "rotulo": "UGs com movimentação",
            "legenda": "Número de UGs com movimentação",
            "tipo": "inteiro",
        },
        "VALOR_SIGILOSO": {
            "rotulo": "Valor sob sigilo",
            "legenda": "Valor identificado como sigiloso (R$)",
            "tipo": "moeda",
        },
        "PCT_SIGILO_VALOR": {
            "rotulo": "Percentual do valor sob sigilo",
            "legenda": "Parcela do valor sob sigilo (%)",
            "tipo": "percentual",
        },
        "VALOR_COM_DATA_TRANSACAO_OBSERVAVEL": {
            "rotulo": "Valor com data da transação observável",
            "legenda": "Valor com DATA TRANSAÇÃO observável (R$)",
            "tipo": "moeda",
        },
        "TAXA_OBSERVABILIDADE_VALOR": {
            "rotulo": "Observabilidade do valor",
            "legenda": "Parcela do valor com DATA TRANSAÇÃO observável (%)",
            "tipo": "percentual",
        },
        "TAXA_OBSERVABILIDADE_REGISTROS": {
            "rotulo": "Observabilidade dos registros",
            "legenda": "Parcela dos registros com DATA TRANSAÇÃO observável (%)",
            "tipo": "percentual",
        },
    },
}


REFERENCIAS_TEMPORAIS = {
    "TRANSACAO": {
        "rotulo": "Ano da transação",
        "descricao": (
            "Operações cuja DATA TRANSAÇÃO é observável."
        ),
        "dataset": agg_transacao_df,
        "metrica_padrao": "VALOR_TRANSACIONADO_OBSERVAVEL",
    },

    "EXTRATO": {
        "rotulo": "Ano do extrato",
        "descricao": (
            "Cobertura temporal pelo extrato; inclui registros "
            "sem data da transação observável."
        ),
        "dataset": agg_extrato_df,
        "metrica_padrao": "VALOR_TOTAL_REGISTRADO",
    },
}


def moeda_br(valor):
    if pd.isna(valor):
        return "—"

    txt = f"{float(valor):,.2f}"

    return (
        "R$ "
        + txt
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def inteiro_br(valor):
    if pd.isna(valor):
        return "—"

    return (
        f"{int(round(float(valor))):,}"
        .replace(",", ".")
    )


def percentual_br(valor):
    if pd.isna(valor):
        return "—"

    return (
        f"{float(valor):.2f}%"
        .replace(".", ",")
    )

In [ ]:
# ============================================================
# 🗺️ FUNÇÃO PRINCIPAL DO MAPA — V1.1
# ============================================================

def gerar_mapa_cpgf_uf(
    ano,
    referencia_temporal="EXTRATO",
    metrica=None,
    salvar=False,
):
    referencia_temporal = str(
        referencia_temporal
    ).upper()

    if referencia_temporal not in REFERENCIAS_TEMPORAIS:
        raise KeyError(
            f"Referência temporal desconhecida: {referencia_temporal}"
        )

    meta_ref = REFERENCIAS_TEMPORAIS[
        referencia_temporal
    ]

    dados_base = meta_ref[
        "dataset"
    ]

    if metrica is None:
        metrica = meta_ref[
            "metrica_padrao"
        ]

    if metrica not in METRICAS_MAPA[
        referencia_temporal
    ]:
        raise KeyError(
            f"Métrica {metrica} não pertence à referência "
            f"{referencia_temporal}."
        )

    ano = int(ano)

    dados = (
        dados_base[
            dados_base["ANO"] == ano
        ]
        .copy()
    )

    if dados.empty:
        raise ValueError(
            f"Não há dados para {ano} em {referencia_temporal}."
        )

    geo = estados_gdf.copy()

    geo["UF"] = (
        geo[COL_UF_GEO]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    colunas_join = [
        c
        for c in dados.columns
        if c != "STATUS_PERIODO"
    ]

    geo = geo.merge(
        dados[
            colunas_join
        ],
        on="UF",
        how="left"
    )

    numeric_cols = [
        c
        for c in colunas_join
        if c not in {
            "UF",
            "ANO",
        }
    ]

    for c in numeric_cols:
        if c in geo.columns:
            geo[c] = (
                pd.to_numeric(
                    geo[c],
                    errors="coerce"
                )
                .fillna(0)
            )

    if COL_NOME_GEO:
        geo["NOME_ESTADO"] = (
            geo[COL_NOME_GEO]
            .astype(str)
        )
    else:
        geo["NOME_ESTADO"] = geo["UF"]

    # --------------------------------------------------------
    # Tooltips específicos por referência
    # --------------------------------------------------------

    if referencia_temporal == "TRANSACAO":
        geo["FMT_VALOR_PRINCIPAL"] = (
            geo["VALOR_TRANSACIONADO_OBSERVAVEL"]
            .apply(moeda_br)
        )
        geo["FMT_PARTICIPACAO"] = (
            geo["PARTICIPACAO_NACIONAL_PCT"]
            .apply(percentual_br)
        )
        geo["FMT_N"] = (
            geo["N_TRANSACOES_OBSERVAVEIS"]
            .apply(inteiro_br)
        )
        geo["FMT_N_UG"] = (
            geo["N_UG_COM_MOVIMENTACAO"]
            .apply(inteiro_br)
        )
        geo["FMT_VALOR_MEDIO"] = (
            geo["VALOR_MEDIO_POR_UG"]
            .apply(moeda_br)
        )
        geo["FMT_COMPRAS"] = (
            geo["VALOR_COMPRAS_OBSERVAVEIS"]
            .apply(moeda_br)
        )
        geo["FMT_SAQUES"] = (
            geo["VALOR_SAQUES_OBSERVAVEIS"]
            .apply(moeda_br)
        )

        tooltip_fields = [
            "NOME_ESTADO",
            "UF",
            "FMT_VALOR_PRINCIPAL",
            "FMT_PARTICIPACAO",
            "FMT_N",
            "FMT_N_UG",
            "FMT_VALOR_MEDIO",
            "FMT_COMPRAS",
            "FMT_SAQUES",
        ]

        tooltip_aliases = [
            "Estado:",
            "UF:",
            "Valor transacionado observável:",
            "Participação nacional:",
            "Transações observáveis:",
            "UGs com movimentação:",
            "Valor médio por UG:",
            "Compras observáveis:",
            "Saques observáveis:",
        ]

        nota_temporal = (
            "Ano da transação: somente registros cuja DATA TRANSAÇÃO "
            "é publicamente observável."
        )

    else:
        geo["FMT_VALOR_PRINCIPAL"] = (
            geo["VALOR_TOTAL_REGISTRADO"]
            .apply(moeda_br)
        )
        geo["FMT_PARTICIPACAO"] = (
            geo["PARTICIPACAO_NACIONAL_PCT"]
            .apply(percentual_br)
        )
        geo["FMT_N"] = (
            geo["N_REGISTROS"]
            .apply(inteiro_br)
        )
        geo["FMT_N_UG"] = (
            geo["N_UG_COM_MOVIMENTACAO"]
            .apply(inteiro_br)
        )
        geo["FMT_VALOR_MEDIO"] = (
            geo["VALOR_MEDIO_POR_UG"]
            .apply(moeda_br)
        )
        geo["FMT_SIGILO"] = (
            geo["VALOR_SIGILOSO"]
            .apply(moeda_br)
        )
        geo["FMT_PCT_SIGILO"] = (
            geo["PCT_SIGILO_VALOR"]
            .apply(percentual_br)
        )
        geo["FMT_OBS_VALOR"] = (
            geo["TAXA_OBSERVABILIDADE_VALOR"]
            .apply(percentual_br)
        )
        geo["FMT_OBS_REG"] = (
            geo["TAXA_OBSERVABILIDADE_REGISTROS"]
            .apply(percentual_br)
        )

        tooltip_fields = [
            "NOME_ESTADO",
            "UF",
            "FMT_VALOR_PRINCIPAL",
            "FMT_PARTICIPACAO",
            "FMT_N",
            "FMT_N_UG",
            "FMT_VALOR_MEDIO",
            "FMT_SIGILO",
            "FMT_PCT_SIGILO",
            "FMT_OBS_VALOR",
            "FMT_OBS_REG",
        ]

        tooltip_aliases = [
            "Estado:",
            "UF:",
            "Valor total registrado:",
            "Participação nacional:",
            "Registros:",
            "UGs com movimentação:",
            "Valor médio por UG:",
            "Valor sob sigilo:",
            "% do valor sob sigilo:",
            "Observabilidade do valor:",
            "Observabilidade dos registros:",
        ]

        nota_temporal = (
            "Ano do extrato: inclui registros cuja DATA TRANSAÇÃO "
            "não é publicamente observável."
        )

    valores = (
        geo[metrica]
        .astype(float)
    )

    vmin = float(
        valores.min()
    )

    vmax = float(
        valores.max()
    )

    if math.isclose(
        vmin,
        vmax
    ):
        vmax = vmin + 1

    escala = (
        linear
        .YlOrRd_09
        .scale(
            vmin,
            vmax
        )
    )

    escala.caption = (
        METRICAS_MAPA[
            referencia_temporal
        ][metrica]["legenda"]
    )

    mapa = folium.Map(
        location=[
            -14.2,
            -51.9,
        ],
        zoom_start=4,
        tiles="CartoDB positron",
        control_scale=True,
    )

    status_periodo = (
        dados[
            "STATUS_PERIODO"
        ].iloc[0]
        if "STATUS_PERIODO" in dados.columns
        else "EXERCICIO_COMPLETO"
    )

    periodo = (
        " — período parcial"
        if status_periodo == "PERIODO_PARCIAL"
        else ""
    )

    titulo = (
        "CPGF por UF da Unidade Gestora"
        f"<br><span style='font-size:14px'>"
        f"{meta_ref['rotulo']} · {ano}{periodo}"
        f" · {METRICAS_MAPA[referencia_temporal][metrica]['rotulo']}"
        "</span>"
    )

    html_titulo = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50%;
        transform: translateX(-50%);
        z-index: 9999;
        background: rgba(255,255,255,0.94);
        padding: 8px 14px;
        border: 1px solid #999;
        border-radius: 6px;
        font-size: 17px;
        font-weight: 600;
        text-align: center;
        ">
        {titulo}
    </div>
    """

    mapa.get_root().html.add_child(
        folium.Element(
            html_titulo
        )
    )

    def style_function(feature):
        valor = (
            feature[
                "properties"
            ]
            .get(
                metrica,
                0
            )
        )

        if valor is None:
            valor = 0

        return {
            "fillColor":
                escala(
                    float(valor)
                ),

            "color":
                "#555555",

            "weight":
                0.8,

            "fillOpacity":
                0.78,
        }

    def highlight_function(feature):
        return {
            "weight": 2.2,
            "color": "#222222",
            "fillOpacity": 0.9,
        }

    folium.GeoJson(
        data=json.loads(
            geo.to_json()
        ),
        name="UF",
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=False,
            sticky=True,
            labels=True,
        ),
    ).add_to(
        mapa
    )

    escala.add_to(
        mapa
    )

    Fullscreen(
        position="topright"
    ).add_to(
        mapa
    )

    nota = f"""
    <div style="
        position: fixed;
        bottom: 18px;
        left: 18px;
        z-index: 9999;
        background: rgba(255,255,255,0.94);
        padding: 7px 10px;
        border: 1px solid #aaa;
        border-radius: 4px;
        font-size: 10px;
        max-width: 430px;
        ">
        <b>Geografia:</b> UF = localização cadastral da Unidade Gestora;
        não representa necessariamente o local físico da transação.
        <br><b>Temporalidade:</b> {nota_temporal}
    </div>
    """

    mapa.get_root().html.add_child(
        folium.Element(
            nota
        )
    )

    if salvar:
        nome = (
            f"mapa_cpgf_uf_"
            f"{referencia_temporal.lower()}_"
            f"{ano}_"
            f"{metrica.lower()}.html"
        )

        caminho = (
            MAPAS_DIR
            / nome
        )

        mapa.save(
            str(caminho)
        )

        print(
            "💾 Mapa salvo:",
            caminho
        )

    return mapa

## 🔟 Mapa de referência

A V1.1 usa como demonstração inicial a visão de **ano do extrato**, pois ela preserva a cobertura temporal do valor registrado, inclusive quando a data da transação não é observável.

O usuário poderá alternar para `ANO_TRANSACAO` no widget seguinte.

In [ ]:
# ============================================================
# 🗺️ MAPA DE REFERÊNCIA V1.1
# ============================================================

REFERENCIA_SELECIONADA = (
    "EXTRATO"
)

ANO_SELECIONADO = 2025

METRICA_SELECIONADA = (
    "VALOR_TOTAL_REGISTRADO"
)

mapa = gerar_mapa_cpgf_uf(
    ano=ANO_SELECIONADO,
    referencia_temporal=REFERENCIA_SELECIONADA,
    metrica=METRICA_SELECIONADA,
    salvar=True,
)

mapa

In [ ]:
# ============================================================
# 🏆 RANKING DO MAPA DE REFERÊNCIA
# ============================================================

dataset_ranking = (
    REFERENCIAS_TEMPORAIS[
        REFERENCIA_SELECIONADA
    ]["dataset"]
)

ranking_df = (
    dataset_ranking[
        dataset_ranking[
            "ANO"
        ]
        == ANO_SELECIONADO
    ]
    .sort_values(
        METRICA_SELECIONADA,
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

ranking_df.index = (
    ranking_df.index
    + 1
)

colunas_ranking = [
    "UF",
    METRICA_SELECIONADA,
    "PARTICIPACAO_NACIONAL_PCT",
    "N_UG_COM_MOVIMENTACAO",
]

if (
    REFERENCIA_SELECIONADA
    == "EXTRATO"
):
    colunas_ranking += [
        "PCT_SIGILO_VALOR",
        "TAXA_OBSERVABILIDADE_VALOR",
    ]

display(
    ranking_df[
        colunas_ranking
    ]
    .head(10)
)

## 1️⃣1️⃣ Seletor interativo no Colab

A interface de teste agora possui três dimensões:

```text
Referência temporal
Ano
Métrica
```

A lista de métricas muda automaticamente quando o usuário alterna entre:

- **Ano da transação**;
- **Ano do extrato**.

Isso impede combinações conceitualmente inválidas, como mostrar `% de sigilo` na população restrita a registros com data da transação observável.

In [ ]:
# ============================================================
# 🎛️ WIDGETS V1.1
# ============================================================

import ipywidgets as widgets


def opcoes_anos_referencia(
    referencia
):
    df = (
        REFERENCIAS_TEMPORAIS[
            referencia
        ]["dataset"]
    )

    anos = sorted(
        df[
            "ANO"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    opcoes = []

    for ano in anos:
        status = (
            df.loc[
                df["ANO"] == ano,
                "STATUS_PERIODO"
            ]
            .iloc[0]
        )

        rotulo = str(
            ano
        )

        if status == "PERIODO_PARCIAL":
            rotulo += " (parcial)"

        opcoes.append(
            (
                rotulo,
                ano,
            )
        )

    return opcoes


def opcoes_metricas_referencia(
    referencia
):
    return [
        (
            info["rotulo"],
            codigo,
        )
        for codigo, info
        in METRICAS_MAPA[
            referencia
        ].items()
    ]


widget_referencia = widgets.Dropdown(
    options=[
        (
            "Ano do extrato — cobertura integral",
            "EXTRATO"
        ),
        (
            "Ano da transação — data observável",
            "TRANSACAO"
        ),
    ],
    value="EXTRATO",
    description="Referência:",
    layout=widgets.Layout(
        width="410px"
    ),
)

widget_ano = widgets.Dropdown(
    options=opcoes_anos_referencia(
        "EXTRATO"
    ),
    value=2025,
    description="Ano:",
    layout=widgets.Layout(
        width="250px"
    ),
)

widget_metrica = widgets.Dropdown(
    options=opcoes_metricas_referencia(
        "EXTRATO"
    ),
    value=REFERENCIAS_TEMPORAIS[
        "EXTRATO"
    ]["metrica_padrao"],
    description="Métrica:",
    layout=widgets.Layout(
        width="470px"
    ),
)

saida = widgets.Output()


def sincronizar_controles(
    change=None
):
    referencia = (
        widget_referencia.value
    )

    op_anos = (
        opcoes_anos_referencia(
            referencia
        )
    )

    anos_validos = [
        valor
        for _, valor
        in op_anos
    ]

    widget_ano.options = (
        op_anos
    )

    if 2025 in anos_validos:
        widget_ano.value = 2025
    elif anos_validos:
        widget_ano.value = (
            anos_validos[-1]
        )

    widget_metrica.options = (
        opcoes_metricas_referencia(
            referencia
        )
    )

    widget_metrica.value = (
        REFERENCIAS_TEMPORAIS[
            referencia
        ]["metrica_padrao"]
    )


def atualizar_mapa(
    change=None
):
    with saida:
        saida.clear_output(
            wait=True
        )

        referencia = (
            widget_referencia.value
        )

        ano = (
            widget_ano.value
        )

        metrica = (
            widget_metrica.value
        )

        mapa = gerar_mapa_cpgf_uf(
            ano=ano,
            referencia_temporal=referencia,
            metrica=metrica,
            salvar=False,
        )

        display(
            mapa
        )

        df = (
            REFERENCIAS_TEMPORAIS[
                referencia
            ]["dataset"]
        )

        top = (
            df[
                df[
                    "ANO"
                ]
                == ano
            ]
            .sort_values(
                metrica,
                ascending=False
            )
            .head(5)
            [
                [
                    "UF",
                    metrica,
                ]
            ]
        )

        print(
            "\nTop 5 UFs na métrica selecionada:"
        )

        display(
            top
        )


widget_referencia.observe(
    sincronizar_controles,
    names="value"
)

widget_referencia.observe(
    atualizar_mapa,
    names="value"
)

widget_ano.observe(
    atualizar_mapa,
    names="value"
)

widget_metrica.observe(
    atualizar_mapa,
    names="value"
)

display(
    widgets.VBox(
        [
            widgets.HBox(
                [
                    widget_referencia,
                    widget_ano,
                ]
            ),
            widget_metrica,
        ]
    )
)

display(
    saida
)

atualizar_mapa()

## 1️⃣2️⃣ Exportação opcional de mapas

A exportação em lote permanece desativada por padrão.

Quando ativada, a V1.1 gera o mapa principal de cada referência:

- `TRANSACAO` → `VALOR_TRANSACIONADO_OBSERVAVEL`;
- `EXTRATO` → `VALOR_TOTAL_REGISTRADO`.

In [ ]:
# ============================================================
# 📤 EXPORTAÇÃO EM LOTE — OPCIONAL
# ============================================================

EXPORTAR_MAPAS_ANUAIS = False

if EXPORTAR_MAPAS_ANUAIS:
    for referencia in [
        "EXTRATO",
        "TRANSACAO",
    ]:
        df = (
            REFERENCIAS_TEMPORAIS[
                referencia
            ]["dataset"]
        )

        metrica = (
            REFERENCIAS_TEMPORAIS[
                referencia
            ]["metrica_padrao"]
        )

        anos = sorted(
            df[
                "ANO"
            ]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        for ano in tqdm(
            anos,
            desc=f"🗺️ {referencia}",
            unit="ano"
        ):
            _ = gerar_mapa_cpgf_uf(
                ano=ano,
                referencia_temporal=referencia,
                metrica=metrica,
                salvar=True,
            )

    print(
        "✅ Mapas anuais exportados."
    )

else:
    print(
        "⏭️ Exportação em lote desativada."
    )

## 1️⃣3️⃣ Contrato metodológico para integração futura

A V1.1 estabelece três camadas derivadas.

```text
CPGF bruto
    │
    ├── CÓDIGO UNIDADE GESTORA
    ▼
dim_ug_geografica
    │
    ├── UF
    ├── FONTE_UF
    ├── TIPO_FONTE_UF
    └── VERSAO_FONTE_UF
    │
    ▼
cpgf_enriquecido_uf_v1_1.parquet
    │
    ├───────────────┐
    ▼               ▼
ANO_TRANSACAO    ANO_EXTRATO_REF
    │               │
    ▼               ▼
agg_cpgf_       agg_cpgf_
uf_ano_         uf_ano_
transacao       extrato
    │               │
    └───────┬───────┘
            ▼
agg_cpgf_uf_ano_dashboard_long
            │
            ▼
Folium / dashboard
```

### Regra de interpretação

**Ano da transação**

> comportamento com data efetiva observável.

**Ano do extrato**

> cobertura pelo ciclo do extrato, inclusive quando a data da transação está protegida.

Não se mistura uma referência com a outra em um mesmo indicador.

---

## Métrica de observabilidade

A V1.1 introduz:

\[
TaxaObservabilidadeValor =
\frac{Valor\ registrado\ com\ DATA\ TRANSAÇÃO\ observável}
{Valor\ total\ registrado\ no\ mesmo\ ano\ do\ extrato}
\times 100
\]

e:

\[
TaxaObservabilidadeRegistros =
\frac{Registros\ com\ DATA\ TRANSAÇÃO\ observável}
{Registros\ totais\ do\ mesmo\ ano\ do\ extrato}
\times 100
\]

Essas métricas descrevem a **observabilidade da base pública**. Elas não são trilhas de irregularidade.

---

## Drill-down futuro

```text
Brasil
  ↓
UF da UG
  ↓
UG
  ↓
fornecedor / portador
  ↓
T01–T09
```

---

## O que não fazer

- não interpretar UF como local físico da compra;
- não misturar `ANO_TRANSACAO` e `ANO_EXTRATO`;
- não apresentar `% sigilo` como se viesse da população restrita a data observável;
- não filtrar a história por `Ativo = SIM` do cadastro mais recente;
- não normalizar por população estadual sem hipótese substantiva;
- não sobrescrever o CSV bruto;
- não transformar a dimensão territorial em T10.

In [ ]:
# ============================================================
# 🧾 METADADOS + CONTRATO TERRITORIAL V1.1
# ============================================================

def sha256_arquivo(
    caminho,
    bloco=1024 * 1024
):
    h = hashlib.sha256()

    with open(
        caminho,
        "rb"
    ) as f:
        while True:
            parte = f.read(
                bloco
            )

            if not parte:
                break

            h.update(
                parte
            )

    return h.hexdigest()


metadata = {
    "data_hora":
        datetime.now()
        .isoformat(),

    "versao_enriquecimento":
        VERSAO_ENRIQUECIMENTO,

    "versao_cadastro_uf":
        VERSAO_CADASTRO_UF,

    "arquivo_cpgf":
        str(
            CPGF_CSV
        ),

    "arquivo_siafi":
        str(
            SIAFI_CSV
        ),

    "sha256_cpgf":
        sha256_arquivo(
            CPGF_CSV
        ),

    "sha256_siafi":
        sha256_arquivo(
            SIAFI_CSV
        ),

    "n_ugs_cpgf":
        int(
            cobertura_df[
                "N_UG_CPGF"
            ].iloc[0]
        ),

    "n_ugs_match_siafi":
        int(
            cobertura_df[
                "N_UG_MATCH_SIAFI"
            ].iloc[0]
        ),

    "n_ugs_match_final":
        int(
            cobertura_df[
                "N_UG_MATCH_FINAL"
            ].iloc[0]
        ),

    "n_complementos_manuais":
        len(
            COMPLEMENTOS_MANUAIS
        ),

    "cobertura_ug_pct":
        float(
            cobertura_df[
                "COBERTURA_UG_PCT"
            ].iloc[0]
        ),

    "cobertura_registros_pct":
        float(
            cobertura_df[
                "COBERTURA_REGISTROS_PCT"
            ].iloc[0]
        ),

    "interpretacao_uf":
        (
            "Localização cadastral da Unidade Gestora; "
            "não representa necessariamente o local físico "
            "da transação."
        ),

    "referencias_temporais": {
        "TRANSACAO":
            (
                "ANO_TRANSACAO derivado de DATA TRANSAÇÃO; "
                "somente registros com data observável."
            ),

        "EXTRATO":
            (
                "ANO_EXTRATO_REF derivado de ANO EXTRATO, "
                "com fallback técnico para COMPETENCIA_ARQUIVO."
            ),
    },

    "geometria_estados":
        (
            f"geobr / estados {ANO_GEOMETRIA_IBGE}"
        ),
}


contrato_territorial = {
    "versao":
        VERSAO_ENRIQUECIMENTO,

    "dimensao_geografica": {
        "chave":
            "UG_ID",

        "formato_chave":
            "string de 6 dígitos",

        "atributo":
            "UF_UG",

        "interpretacao":
            metadata[
                "interpretacao_uf"
            ],

        "fontes": [
            "SIAFI_DADOS_UG_2025",
            "COMPLEMENTO_MANUAL_2026_08_13",
        ],

        "cobertura_esperada_base_atual":
            "100%",
    },

    "referencias_temporais": {
        "TRANSACAO": {
            "campo_ano":
                "ANO_TRANSACAO",

            "cobertura":
                "somente registros com DATA TRANSAÇÃO observável",

            "metrica_principal":
                "VALOR_TRANSACIONADO_OBSERVAVEL",

            "uso":
                (
                    "comportamento temporal, compras, saques "
                    "e integração com trilhas"
                ),
        },

        "EXTRATO": {
            "campo_ano":
                "ANO_EXTRATO_REF",

            "cobertura":
                (
                    "registros atribuíveis ao ano do extrato, "
                    "inclusive sem DATA TRANSAÇÃO observável"
                ),

            "metrica_principal":
                "VALOR_TOTAL_REGISTRADO",

            "uso":
                (
                    "cobertura integral, sigilo "
                    "e observabilidade"
                ),
        },
    },

    "agregados": {
        "transacao":
            str(
                AGG_TRANSACAO_PARQUET
            ),

        "extrato":
            str(
                AGG_EXTRATO_PARQUET
            ),

        "dashboard_long":
            str(
                AGG_DASHBOARD_PARQUET
            ),
    },

    "salvaguardas": [
        (
            "UF da UG não representa necessariamente "
            "o local físico da transação."
        ),
        (
            "Não combinar ANO_TRANSACAO e ANO_EXTRATO "
            "em uma única métrica temporal."
        ),
        (
            "Sigilo e observabilidade devem ser analisados "
            "na referência do extrato."
        ),
        (
            "A dimensão geográfica não constitui trilha "
            "de irregularidade."
        ),
    ],
}


METADATA_JSON = (
    CONTROLE_DIR
    / "metadata_enriquecimento_geografico_v1_1.json"
)

CONTRATO_JSON = (
    CONTROLE_DIR
    / "contrato_territorial_v1_1.json"
)

with open(
    METADATA_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(
    CONTRATO_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        contrato_territorial,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "✅ Metadados:"
)

print(
    METADATA_JSON
)

print(
    "✅ Contrato territorial:"
)

print(
    CONTRATO_JSON
)

display(
    pd.DataFrame(
        [metadata]
    ).T
)

# ✅ Encerramento — Enriquecimento Geográfico V1.1

Ao final da execução, os principais produtos serão:

```text
Analise_Geografica_CPGF_v1_1/
├── 00_controle/
│   ├── metadata_enriquecimento_geografico_v1_1.json
│   ├── contrato_territorial_v1_1.json
│   ├── catalogo_metricas_territoriais.csv
│   ├── resumo_observabilidade_ano_extrato.csv
│   ├── resumo_ug_exterior.csv
│   └── ugs_sem_uf.csv
│
├── 01_dimensoes/
│   ├── dim_ug_geografica.parquet
│   └── dim_ug_geografica.csv
│
├── 02_enriquecido/
│   └── cpgf_enriquecido_uf_v1_1.parquet
│
├── 03_agregados/
│   ├── agg_cpgf_uf_ano_transacao.parquet
│   ├── agg_cpgf_uf_ano_transacao.csv
│   ├── agg_cpgf_uf_ano_extrato.parquet
│   ├── agg_cpgf_uf_ano_extrato.csv
│   ├── agg_cpgf_uf_ano_dashboard_long.parquet
│   └── agg_cpgf_uf_ano_dashboard_long.csv
│
└── 04_mapas/
    └── mapa_cpgf_uf_<referencia>_<ano>_<metrica>.html
```

## Critérios de sucesso da V1.1

1. `UG_ID` permanece com seis dígitos.
2. cobertura UG→UF permanece em 100% na base atual.
3. cinco complementos continuam rastreáveis.
4. `ANO_TRANSACAO` e `ANO_EXTRATO_REF` permanecem separados.
5. a visão por transação reproduz o universo observável da V1.0, com nomenclatura corrigida.
6. a visão por extrato recompõe o valor positivo sem ajustes no mesmo universo geográfico.
7. o valor sob sigilo é mensurável na visão por extrato.
8. observabilidade do valor e dos registros é explicitamente mensurada.
9. `N_UG_COM_MOVIMENTACAO` substitui a nomenclatura ambígua `N_UG_ATIVAS`.
10. o Folium impede combinações inválidas entre referência temporal e métrica.
11. arquivos brutos permanecem imutáveis.
12. a camada geográfica continua separada das regras T01–T09.

Se esses critérios forem atendidos na execução real, a dimensão territorial poderá ser congelada como **Enriquecimento Geográfico 1.1.0** e incorporada à arquitetura futura de `src/`.